In [ ]:
# Install dependencies for standard PyTorch and PEFT execution
!pip install -q transformers peft accelerate bitsandbytes datasets numpy

In [ ]:
import torch
import numpy as np
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

# Configuration
MODEL_ID = "google/gemma-4-e4b"
LORA_RANK = 16
LORA_ALPHA = 32
POPULATION_SIZE = 32
LEARNING_RATE = 0.05
NOISE_STD = 0.02
GENERATIONS = 50
BATCH_SIZE = 4

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Load Base Model in 4-bit precision
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.bfloat16
)

# Inject LoRA Adapters
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

In [ ]:
essa_params = []

def extract_and_reparameterize(module, rank):
    """Extracts SVD from LoRA matrices and returns tracking dictionary."""
    # PEFT stores weights in base_layer (for bnb) and lora_A.default.weight
    A = module.lora_A.default.weight.data.float()
    B = module.lora_B.default.weight.data.float()

    # Compute combined matrix W = BA
    W = B @ A

    # SVD Decomposition W = U S V^T
    U, S, V = torch.svd(W)

    # Truncate to desired rank
    U_r = U[:, :rank]
    S_r = S[:rank]
    V_r = V[:, :rank]

    # Only S will be optimized via Evolution Strategies
    U_r.requires_grad = False
    V_r.requires_grad = False
    S_r.requires_grad = True

    return {"module": module, "U": U_r, "S": S_r, "V": V_r}

def apply_singular_values(param_dict, current_S):
    """Reconstructs LoRA matrices from singular values and injects them back."""
    U, V, module = param_dict["U"], param_dict["V"], param_dict["module"]

    # Prevent NaNs by ensuring S is positive before square root
    S_safe = torch.relu(current_S) + 1e-8
    S_sqrt = torch.diag(torch.sqrt(S_safe))

    # Factor S back into B and A
    B_new = U @ S_sqrt
    A_new = S_sqrt @ V.T

    # Update model weights in place
    module.lora_B.default.weight.data = B_new.to(module.lora_B.default.weight.dtype)
    module.lora_A.default.weight.data = A_new.to(module.lora_A.default.weight.dtype)

# Isolate SVD components for all LoRA layers
for name, module in model.named_modules():
    if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
        essa_params.append(extract_and_reparameterize(module, LORA_RANK))

print(f"Isolated {len(essa_params)} matrices for SVD optimization.")

In [ ]:
# Load a subset of GSM8K for evaluation
dataset = load_dataset("openai/gsm8k", "main", split="train[:1000]")

def get_batch(batch_size):
    indices = np.random.choice(len(dataset), batch_size, replace=False)
    return [dataset[int(i)] for i in indices]

def format_prompt(question):
    return f"Solve this math problem. State the final numerical answer at the end.\nQuestion: {question}\nAnswer: "

def evaluate_reward(response, ground_truth):
    """Extracts terminal numbers and performs exact match."""
    predictions = re.findall(r'-?\d+', response.replace(",", ""))
    truths = re.findall(r'-?\d+', ground_truth.split("####")[-1].replace(",", ""))

    if not predictions or not truths:
        return 0.0
    return 1.0 if predictions[-1] == truths[-1] else 0.0

In [ ]:
# Main Training Loop
for generation in range(GENERATIONS):
    # 1. Generate Population Noise
    population_noise = []
    for _ in range(POPULATION_SIZE):
        noise_layer = [torch.randn_like(p["S"]) * NOISE_STD for p in essa_params]
        population_noise.append(noise_layer)

    rewards = np.zeros(POPULATION_SIZE)
    eval_batch = get_batch(BATCH_SIZE)

    # 2. Evaluate Population (Forward Pass Only)
    for p_idx, noise in enumerate(population_noise):

        # Apply perturbed singular values: S' = S + noise
        for i, param in enumerate(essa_params):
            perturbed_S = param["S"] + noise[i]
            apply_singular_values(param, perturbed_S)

        # Run inference on the batch
        batch_rewards = []
        for sample in eval_batch:
            inputs = tokenizer(format_prompt(sample["question"]), return_tensors="pt").to("cuda")

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=128,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id
                )

            response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            batch_rewards.append(evaluate_reward(response, sample["answer"]))

        rewards[p_idx] = np.mean(batch_rewards)

    # 3. Standardize Rewards (Fitness Baseline)
    mean_reward = np.mean(rewards)
    std_reward = np.std(rewards) + 1e-8
    normalized_rewards = (rewards - mean_reward) / std_reward

    print(f"Generation {generation + 1}/{GENERATIONS} | Mean Reward: {mean_reward:.4f} | Max Reward: {np.max(rewards):.4f}")

    # 4. Parameter Update (ES Gradient Estimator)
    with torch.no_grad():
        for i, param in enumerate(essa_params):
            grad_estimator = torch.zeros_like(param["S"])
            for p_idx in range(POPULATION_SIZE):
                grad_estimator += normalized_rewards[p_idx] * population_noise[p_idx][i]

            grad_estimator /= (POPULATION_SIZE * NOISE_STD)

            # Apply gradient ascent to the singular values
            param["S"].data += LEARNING_RATE * grad_estimator

            # Reconstruct central matrices for the next generation
            apply_singular_values(param, param["S"])

In [ ]:
# Save the final LoRA adapters
# The weights are already correctly synchronized via apply_singular_values in the final step of the loop
output_dir = "./gemma4-e4b-essa-gsm8k"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"ESSA optimized adapters saved to {output_dir}")